In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw

In [ ]:
import os
from pathlib import Path
import sys

sys.path.append(str(Path(os.getcwd()).parent / "src"))

import importlib
import utils
import active_gel_fem
import myosin_actin_gel_fem

importlib.reload(active_gel_fem)
importlib.reload(myosin_actin_gel_fem)
importlib.reload(utils)

from myosin_actin_gel_fem import MyosinActinGelDynBeta1D, MyosinActinGelDynBeta2D
from utils import tanh_source, sharp_source, animate_2d

# Active gel with myosin-dependent nematic relaxation

This notebook explores a generalisation of `actin_myo_gel.ipynb` where the nematic relaxation
rates **β₁** and **β₂** depend on the local myosin concentration.

**Convention change for β₂**: β₂ is now the nematic relaxation *rate* (= 1 / old β₂), so:
- large β₂ → actin reorients quickly  
- β₂ → 0   → actin orientation frozen (heavily cross-linked by myosin)

A natural choice that satisfies β₂ → 0 as myosin increases:
$$
\beta_2(m) = \frac{\beta_2^0}{1 + (m / m_{\beta})^{n_\beta}}
$$
and analogously for β₁.  At zero myosin the rates equal their bare values; as myosin grows
the rates decay, freezing the nematic order.

## 1D simulation

In [ ]:
# Myosin-dependent relaxation rates
# beta2_0 and beta1_0 are the bare rates at m = 0
# m_beta sets the myosin scale at which the rates drop to half
# n_beta is the Hill cooperativity

beta1_0 = 1.0
beta2_0 = 1.0
m_beta  = 0.3
n_beta  = 2

def beta1_func(m):
    """β₁(m): nematic-velocity coupling rate, decreases with myosin."""
    r = m / m_beta
    return beta1_0 / (1 + r**n_beta)

def beta2_func(m):
    """β₂(m): nematic relaxation rate, → 0 as actin gets cross-linked."""
    r = m / m_beta
    return beta2_0 / (1 + r**n_beta)

In [ ]:
T   = 50
tau = 0.01

S    = sharp_source(center=0.5, width=0.1, value=1, axis='x') + 0.1
m0   = sharp_source(center=0.5, width=0.1, value=1.0, axis='x')
Qsq  = sharp_source(center=0.5, width=0.1, value=2.0, axis='x') - 1

sim1d = MyosinActinGelDynBeta1D(
    beta1_func = beta1_func,
    beta2_func = beta2_func,
    maxh  = 0.01,
    # mechanics
    gamma = 1.0,
    eta_1 = 1,
    eta_2 = 0,
    chi0  = 0.01,
    chi1  = 0.5,
    # nematic (bare values; actual rates are beta_func(m))
    kappa = 1e-3,
    beta1 = beta1_0,
    beta2 = beta2_0,
    Qsq   = Qsq,
    # density / myosin
    S      = S,
    m0     = m0,
    k0     = 0.2,
    k1     = 1,
    k_m    = 1,
    n_hill = 4,
    m_ref  = 0.1,
    D_rho  = 1e-4,
    D_m    = 1e-4,
    # base
    k     = 0.5,
    D     = 1e-3,
    rho0  = 1.0,
)

sim1d.simulate(tend=T, tau=tau, save_interval=int(0.1/tau))
data1d = sim1d.export(n_samples=200)

In [ ]:
# Show how beta1 and beta2 vary across the domain at steady state
xc = data1d['x']
m_ss = data1d['m'][-1]

r = m_ss / m_beta
b1_ss = beta1_0 / (1 + r**n_beta)
b2_ss = beta2_0 / (1 + r**n_beta)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].plot(xc, m_ss);   axes[0].set_title(r'$m$ (myosin, steady state)')
axes[1].plot(xc, b1_ss);  axes[1].set_title(r'$\beta_1(m)$')
axes[2].plot(xc, b2_ss);  axes[2].set_title(r'$\beta_2(m)$')
for ax in axes:
    ax.set_xlabel('x')
    ax.axvline(0.4, color='k', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(0.6, color='k', ls='--', lw=0.8, alpha=0.5)
plt.suptitle('Myosin-dependent relaxation rates at steady state')
plt.tight_layout()
plt.show()

In [ ]:
# Kymograph: rho, m, v, Q over time
fig, axes = plt.subplots(1, 4, figsize=(10, 4))

fields = [
    (data1d['rho'], r'$\rho$ (actin)',   'inferno'),
    (data1d['m'],   r'$m$ (myosin)',     'viridis'),
    (data1d['v'],   r'$v$',              'RdBu_r'),
    (data1d['Q'],   r'$Q_{xx}$',         'RdBu_r'),
]

t_end = data1d['rho'].shape[0] * 0.1

for ax, (field, title, cmap) in zip(axes, fields):
    vmax = np.max(np.abs(field))
    vmin = -vmax if cmap == 'RdBu_r' else 0
    im = ax.imshow(field, aspect='auto', interpolation='none',
                   origin='upper', cmap=cmap, extent=(0, 1, t_end, 0),
                   vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, location='bottom', pad=0.12,
                 ticks=[vmin, vmax], format='%.1g').set_label(title, fontsize=12)
    ax.set_title(title)
    ax.set_xlabel('Position')

axes[0].set_ylabel('Time')
plt.suptitle('1D Dynamic-β Myosin–Actin Gel  —  kymographs', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Steady-state profiles
xc = data1d['x']
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, (field, label) in zip(axes, [
    (data1d['rho'], r'$\rho$'),
    (data1d['m'],   r'$m$'),
    (data1d['v'],   r'$v$'),
    (data1d['Q'],   r'$Q_{xx}$'),
]):
    ax.plot(xc, field[-1])
    ax.set_xlabel('x')
    ax.set_title(label)
    ax.axvline(0.4, color='k', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(0.6, color='k', ls='--', lw=0.8, alpha=0.5)

plt.suptitle('Steady-state profiles (dashed = source band edges)')
plt.tight_layout()
plt.show()

## 2D simulation

In [ ]:
T   = 100
tau = 0.05
n_samples = 120
width  = 0.05
center = 0.5
Lx = 1
Ly = 1

source = (tanh_source(center=center, width=width, value=1, axis='x', interface_length=0.005)
          * tanh_source(center=center, width=Ly*0.9, value=1, axis='y', interface_length=0.1))

S    = 0.8 * source + 0.2
m0   = source
Qsq  = 2 * source - 1

sim2d = MyosinActinGelDynBeta2D(
    beta1_func = beta1_func,
    beta2_func = beta2_func,
    width=Lx, height=Ly, maxh=0.03,
    # mechanics
    gamma = 1,
    eta_1 = 1,
    eta_2 = 0,
    chi0  = 0.05,
    chi1  = 1,
    # nematic (bare values; actual rates are beta_func(m))
    kappa = 1e-4,
    beta1 = beta1_0,
    beta2 = beta2_0,
    Qsq   = Qsq,
    # density / myosin
    S      = S,
    m0     = m0,
    k0     = 0.2,
    k1     = 0.8,
    k_m    = 1,
    n_hill = 4,
    m_ref  = 0.2,
    D_rho  = 1e-4,
    D_m    = 1e-4,
    # base
    k     = 0.5,
    D     = 1e-3,
    rho0  = 1.0,
)

sim2d.simulate(tend=T, tau=tau, save_interval=int(0.5/tau))
data2d = sim2d.export(n_samples=n_samples)

In [ ]:
from utils import add_cbar, plot_2d_frame, animate_2d
import matplotlib.colors as mcolors

inferno_80 = mcolors.ListedColormap(plt.get_cmap('inferno')(np.linspace(0, 0.7, 256)))
plot_2d_frame(data2d, t=-1, title=f't = {T}', cmap_rho=inferno_80)

In [ ]:
# Mid-line (y=0.5) kymographs
n_mid = data2d['rho'].shape[2] // 2
left  = 20
right = 100

compression = -np.gradient(data2d['vx'], axis=1) * n_samples

plt.rcParams.update({'font.size': 15})
fig, axes = plt.subplots(1, 5, figsize=(16, 6))
fields = [
    (data2d['rho'][1:, left:right, n_mid], r'$\rho$',  inferno_80),
    (data2d['m'][1:,   left:right, n_mid], r'$m$',     'viridis'),
    (data2d['vx'][1:,  left:right, n_mid], r'$v_x$',   'RdBu_r'),
    (compression[1:,   left:right, n_mid], r'$-\partial_x v_x$', 'RdBu_r'),
    (-data2d['Q'][1:,  left:right, n_mid], r'$Q_{yy}$','RdBu_r'),
]
t_end = data2d['rho'].shape[0] * 0.5

for ax, (field, title, cmap) in zip(axes, fields):
    vmax = np.max(np.abs(field))
    vmin = -vmax if cmap == 'RdBu_r' else 0
    im = ax.imshow(field, aspect='auto', interpolation='none',
                   origin='upper', cmap=cmap, extent=(0, 1, t_end, 0),
                   vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, location='bottom', pad=0.15,
                 ticks=[vmin, vmax], format='%.2g')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_xticks([])

axes[0].set_ylabel('Time')
plt.suptitle('2D Dynamic-β  —  mid-line kymographs', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
animate_2d(data2d,
           filename=f'../figures/actin_myo_dynbeta_chi0_{sim2d.chi0}_chi1_{sim2d.chi1}.mp4',
           n_arrows=10, fps=6, dt=0.5,
           title_fmt='t = {t:.1f}', cmap_rho=inferno_80)